In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/01011
0


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  697.2510786929424
RUN  2 , total integrated cost =  427.6010747831584
RUN  3 , total integrated cost =  202.70926225237986
RUN  4 , total integrated cost =  57.38099947702033
RUN  5 , total integrated cost =  54.42596989853247
RUN  6 , total integrated cost =  50.458266597736134
RUN  7 , total integrated cost =  48.90824966035195
RUN  8 , total integrated cost =  47.07886672354446
RUN  9 , total integrated cost =  46.02286747009303
RUN  10 , total integrated cost =  44.827443467610706
RUN  11 , total integrated cost =  43.97750054212841
RUN  12 , total integrated cost =  43.061493042090994
RUN  13 , total integrated cost =  42.43262849504851
RUN  14 , total integrated cost =  41.793202713660335
RUN  15 , total integrated cost =  41.315890267633065
RUN  1

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  146 , total integrated cost =  33.35863957653799
Improved over  146  iterations in  22.906567523255944  seconds by  99.34556125508132  percent.
Problem in initial value trasfer:  Vmean_exc -56.624466399857866 -56.62446631755871
weight =  1528.0268898569777
set cost params:  1.0 1528.0268898569777 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5076.586509233994
Gradient descend method:  None
RUN  1 , total integrated cost =  4627.852027622997
RUN  2 , total integrated cost =  4624.642815430091
RUN  3 , total integrated cost =  4622.744411317346
RUN  4 , total integrated cost =  4622.660967654475
RUN  5 , total integrated cost =  4617.0152879874595
RUN  6 , total integrated cost =  4615.436791119432
RUN  7 , total integrated cost =  4615.415719388184
RUN  8 , total integrated cost =  4615.2751395650475
RUN  9 , total integrated cost =  4614.693069681605
RUN  10 , total integrated cost =  4614.622530151184
RUN  11 , total 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  439 , total integrated cost =  4409.5602849148845
Improved over  439  iterations in  29.367648942396045  seconds by  13.139266377236552  percent.
Problem in initial value trasfer:  Vmean_exc -56.62684393858193 -56.62681578933133
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13018.074640346456
Gradient descend method:  None
RUN  1 , total integrated cost =  803.3679377776556
RUN  2 , total integrated cost =  140.16858885993747
RUN  3 , total integrated cost =  60.67008751585198
RUN  4 , total integrated cost =  52.03587319697629
RUN  5 , total integrated cost =  51.88512131488117
RUN  6 , total integrated cost =  51.8513515139626
RUN  7 , total integrated cost =  51.79854950880647
RUN  8 , total integrated cost =  51.77280230520265
RUN  9 , total integrated cost =  51.69842142009918
RUN  10 , total integrated cost =  51.64743950890257
RUN  11 ,

...

RUN  1 , total integrated cost =  23532.63614302554
Control only changes marginally.
RUN  1 , total integrated cost =  23532.63614302554
Improved over  1  iterations in  0.1438909787684679  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -73.65794421761755 -73.65794430566882
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  1 , total integrated cost =  33290.047972975575
RUN  2 , total integrated cost =  33290.04797297555


ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  33290.04797297555
Control only changes marginally.
RUN  3 , total integrated cost =  33290.04797297555
Improved over  3  iterations in  0.31378174014389515  seconds by  1.049533422303739e-05  percent.
Problem in initial value trasfer:  Vmean_exc -68.82426086952952 -68.82427979279365
weight =  10.000001049533532
set cost params:  1.0 10.000001049533532 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.04797297592
Gradient descend method:  None
RUN  1 , total integrated cost =  33290.04797297591


ERROR:root:Problem in initial value trasfer


RUN  2 , total integrated cost =  33290.04797297591
Control only changes marginally.
RUN  2 , total integrated cost =  33290.04797297591
Improved over  2  iterations in  0.25884594582021236  seconds by  2.842170943040401e-14  percent.
Problem in initial value trasfer:  Vmean_exc -68.82426086953004 -68.82427979279416


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
found solution for  10
------- 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  72 , total integrated cost =  124.64876408766358
Improved over  72  iterations in  8.001381916925311  seconds by  98.49610598608055  percent.
Problem in initial value trasfer:  Vmean_exc -56.6397619733967 -56.639762420021555
weight =  660.4082504724043
set cost params:  1.0 660.4082504724043 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8187.359735090244
Gradient descend method:  None
RUN  1 , total integrated cost =  8050.293391365281
RUN  2 , total integrated cost =  8049.175958154019
RUN  3 , total integrated cost =  8049.0436418981135
RUN  4 , total integrated cost =  8047.089025005549
RUN  5 , total integrated cost =  8045.392401522051
RUN  6 , total integrated cost =  8045.025902061496
RUN  7 , total integrated cost =  8044.445264746426
RUN  8 , total integrated cost =  8044.335721552452
RUN  9 , total integrated cost =  8029.2381936795555
RUN  10 , total integrated cost =  8022.199214754519
RUN  11 , total integ

ERROR:root:Problem in initial value trasfer


RUN  30 , total integrated cost =  8016.301079965957
Control only changes marginally.
RUN  30 , total integrated cost =  8016.301079965957
Improved over  30  iterations in  4.059437597170472  seconds by  2.0893018098514204  percent.
Problem in initial value trasfer:  Vmean_exc -56.6386314878368 -56.63864806572297
-------  30 0.4250000000000001 0.5250000000000002
found solution for  30
-------  35 0.5500000000000003 0.5250000000000002
found solution for  35
-------  40 0.5250000000000001 0.5500000000000003
found solution for  40
-------  45 0.5000000000000002 0.5750000000000003
[0, 5, 10, 15, 20, 30, 35, 40] []
closest index  40
set cost params:  1.0 10.0 0.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  20682.357026508653
Gradient descend method:  None
RUN  1 , total integrated cost =  20627.917026235024
RUN  2 , total integrated cost =  20627.907067412594
RUN  3 , total integrated cost =  20627.90706508616
RUN  4 , total integrated cost 

...

RUN  50 , total integrated cost =  20600.10534240081
Control only changes marginally.
RUN  51 , total integrated cost =  20600.10534240081
Improved over  51  iterations in  3.305868646129966  seconds by  0.10315887327728035  percent.
Problem in initial value trasfer:  Vmean_exc -56.696420865029616 -56.696421028033114
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
[0, 5, 10, 15, 20, 30, 35, 40, 50, 55, 60, 70, 75, 80, 90, 100, 105, 110, 120, 130, 140, 25] [60]
closest index  80
set cost params:  1.0 10.0 0.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  247.7191518066573
Gradient descend method:  None
RUN  1 , total integrated cost =  232.6411597768337
RUN  2 , total integrated cost =  232.40837060610434
RUN  3 , total integrated cost =  232.38942192645493
RUN  4 , total integrated cost =

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  377 , total integrated cost =  228.68888621954682
Improved over  377  iterations in  25.394373374059796  seconds by  7.682193907220963  percent.
Problem in initial value trasfer:  Vmean_exc -56.695183353770744 -56.695183367113735
weight =  877.6602766081306
set cost params:  1.0 877.6602766081306 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  20066.409668479224
Gradient descend method:  None
RUN  1 , total integrated cost =  20053.734236907632
RUN  2 , total integrated cost =  20053.733091640504
RUN  3 , total integrated cost =  20053.73308116885
RUN  4 , total integrated cost =  20053.73308116871


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  20053.73308116869
RUN  6 , total integrated cost =  20053.73308116869
Control only changes marginally.
RUN  6 , total integrated cost =  20053.73308116869
Improved over  6  iterations in  0.6043216437101364  seconds by  0.06317317108522502  percent.
Problem in initial value trasfer:  Vmean_exc -56.69517814191014 -56.69517832172776
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
found solution for  85
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.7500000000000004
found solution for  95
-------  100 0.4500000000000001 0.7750000000000005
-------  105 0.5750000000000002 0.7750000000000005
-------  110 0.5000000000000002 0.8000000000000005
-------  115 0.4250000000000001 0.8250000000000005
found solution for  115
-------  120 0.5500000000000003 0.8250000000000005
-------

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  1765.342995588272
set cost params:  1.0 1765.342995588272 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5076.986704256988
Gradient descend method:  None
RUN  1 , total integrated cost =  5076.003242826418
RUN  2 , total integrated cost =  5075.999340459123
RUN  3 , total integrated cost =  5075.998808806369
RUN  4 , total integrated cost =  5075.

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  96 , total integrated cost =  5064.792683242294
Improved over  96  iterations in  6.412673730403185  seconds by  0.24018225228893186  percent.
Problem in initial value trasfer:  Vmean_exc -56.62715927188208 -56.62712915094471
-------  10 0.4250000000000001 0.42500000000000016
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  3997.6201203531505
set cost params:  1.0 3997.6201203531505 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13013.266411592758
Gradient descend method:  None
RUN  1 , total integrated cost =  13013.261540579784
RUN  2 , total integrated cost =  13013.261540579766
RUN  3 , total integrated cost =  13013.261540579762
RUN  4 , total integrated cost =  13013.261540579753
RUN  5 , total integrated cost =  13013.261540579751


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  13013.261540579748
RUN  7 , total integrated cost =  13013.261540579748
Control only changes marginally.
RUN  7 , total integrated cost =  13013.261540579748
Improved over  7  iterations in  0.8691193033009768  seconds by  3.74311326396537e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.67042201816924 -56.67042800363234
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  677.1705666928381
set cost params:  1.0 677.1705666928381 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8218.724343272386
Gradient descend method:  None
RUN  1 , total integrated cost =  8218.71786508881
RUN  2 , total integrated cost =  8218.717861981357
RUN  3 , total integrated cost =  8218.717861947785
RUN  4 , total integrated cost =  8218.717861947707
RUN  5 , total integrated cost =  8218.717861947694
RUN  6 , total integrated cost =  8218.71786

...

RUN  1 , total integrated cost =  34485.91687506635
RUN  2 , total integrated cost =  34485.91687506635
Control only changes marginally.
RUN  2 , total integrated cost =  34485.91687506635
Improved over  2  iterations in  0.34624034725129604  seconds by  7.105427357601002e-14  percent.
Problem in initial value trasfer:  Vmean_exc -56.70312087650492 -56.703120782118276
-------  80 0.5250000000000001 0.7000000000000004
converged for  80
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  90 0.6000000000000003 0.7250000000000004
converged for  90
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  100 0.4500000000000001 0.7750000000000005
converged for  100
-------  105 0.5750000000000002 0.7750000000000005
converged for  105
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  120 0.5500000000000003 0.8250000000000005
converged for  120
------

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5094.421237469255
Control only changes marginally.
RUN  1 , total integrated cost =  5094.421237469255
Improved over  1  iterations in  0.28993214294314384  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.62717660684359 -56.62714637555054
-------  10 0.4250000000000001 0.42500000000000016
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  30 0.4250000000000001 0.5250000000000002
converged for  30
-------  35 0.5500000000000003 0.5250000000000002
converged for  35
-------  40 0.5250000000000001 0.5500000000000003
converged for  40
-------  45 0.5000000000000002 0.5750000000000003
converged for  45
-------  50 0.47500000000000014 0.6000000000000003
converged for  50
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  60 0.5500

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  34485.91687506635
Control only changes marginally.
RUN  1 , total integrated cost =  34485.91687506635
Improved over  1  iterations in  0.19379601255059242  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70312087650492 -56.703120782118276
-------  80 0.5250000000000001 0.7000000000000004
converged for  80
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  90 0.6000000000000003 0.7250000000000004
converged for  90
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  100 0.4500000000000001 0.7750000000000005
converged for  100
-------  105 0.5750000000000002 0.7750000000000005
converged for  105
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  120 0.5500000000000003 0.8250000000000005
converged for  120
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-----

In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [ ]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  149.4589024020993
Gradient descend method:  None
RUN  1 , total integrated cost =  25.38010978899401
RUN  2 , total integrated cost =  25.283039271231374
RUN  3 , total integrated cost =  25.237874094608717
RUN  4 , total integrated cost =  25.210031973210832
RUN  5 , total integrated cost =  25.169162691011305
RUN  6 , total integrated cost =  25.136474999472128
RUN  7 , total integrated cost =  25.110694242987076
RUN  8 , total integrated cost =  25.08930029624334
RUN  9 , total integrated cost =  25.07730130034309
RUN  10 , total integrated cost =  25.067401427145175
RUN  11 , total integrated cost =  25.06053800841651
RUN  12 , total integrated cost =  25.055026013083364
RUN  13 , total integrated cost =  25.04272157184986
RUN  14 , total integrated cost =  25.03506592437172
RUN  15 , total integrated cost =  25.011910369398155
RUN

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  95 , total integrated cost =  5075.02320551564
Improved over  95  iterations in  17.1262067258358  seconds by  0.36311775531044077  percent.
Problem in initial value trasfer:  Vmean_exc -56.62454091653633 -56.624538815102866
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  74.96461940279714
Gradient descend method:  None
RUN  1 , total integrated cost =  26.054215686167318
RUN  2 , total integrated cost =  26.03141557991416
RUN  3 , total integrated cost =  26.019739665326657
RUN  4 , total integrated cost =  25.956293508786302
RUN  5 , total integrated cost =  25.95067394326626
RUN  6 , total integrated cost =  25.95002203507119
RUN  7 , total integrated cost =  25.94963484989465
RUN  8 , total integrated cost =  25.9496121025085
RUN  9 , total integrated cost =  25.949603286835277
RUN  10 , total integrated cost =  25.949556625073956
RUN  11 , 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  95 , total integrated cost =  25.94893800694548
Improved over  95  iterations in  15.275246674194932  seconds by  65.38508670668011  percent.
Problem in initial value trasfer:  Vmean_exc -56.67067509138767 -56.67067507047604
weight =  100
set cost params:  1.0 100.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  331.7494758984014
Gradient descend method:  HS
RUN  1 , total integrated cost =  327.9111952853329
RUN  2 , total integrated cost =  327.39076392680164
RUN  3 , total integrated cost =  325.91604931405294
RUN  4 , total integrated cost =  322.7820841835824
RUN  5 , total integrated cost =  322.47879060577526
RUN  6 , total integrated cost =  322.2379254478898
RUN  7 , total integrated cost =  321.861674178995
RUN  8 , total integrated cost =  321.26191320578135
RUN  9 , total integrated cost =  318.9741400071743
RUN  10 , total integrated cost =  318.55619188057807
RUN  11 , total integrated cost =  317.40639357

...

RUN  1000 , total integrated cost =  16395.16683215136
RUN  1000 , total integrated cost =  16395.16683215136
Improved over  1000  iterations in  143.50050134398043  seconds by  20.484348239587135  percent.
Problem in initial value trasfer:  Vmean_exc -56.69642287348715 -56.69642297195807
-------  65 0.5000000000000002 0.6500000000000004
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  70.63285309704989
Gradient descend method:  None
RUN  1 , total integrated cost =  67.64566643451506
RUN  2 , total integrated cost =  67.64559095634579
RUN  3 , total integrated cost =  67.64559092141857
RUN  4 , total integrated cost =  67.64559092095855
RUN  5 , total integrated cost =  67.64559092094666
RUN  6 , total integrated cost =  67.64559092094642
RUN  7 , total integrated cost =  67.6455909209464
RUN  8 , total integrated cost =  67.6455909209464
Control only changes marginally.
RUN  8 , total integrated cost =  67.6455909209464
Improved o

ERROR:root:Problem in initial value trasfer


RUN  1000 , total integrated cost =  1724.4673046067958
RUN  1000 , total integrated cost =  1724.4673046067958
Improved over  1000  iterations in  199.3038672953844  seconds by  24.609815431385286  percent.
Problem in initial value trasfer:  Vmean_exc -56.69517379540978 -56.695174352299524
weight =  1162.902328565272
set cost params:  1.0 1162.902328565272 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  20053.764711675016
Gradient descend method:  None
RUN  1 , total integrated cost =  20049.204961437503
RUN  2 , total integrated cost =  20048.048448480044
RUN  3 , total integrated cost =  20046.279372214616
RUN  4 , total integrated cost =  20045.165761821452
RUN  5 , total integrated cost =  20043.294813078806
RUN  6 , total integrated cost =  20042.090753487573
RUN  7 , total integrated cost =  20040.07907187142
RUN  8 , total integrated cost =  20038.642684979885
RUN  9 , total integrated cost =  20035.719548481065
RUN  10 , total integrated cost =  200

ERROR:root:Problem in initial value trasfer


RUN  1000 , total integrated cost =  13239.628311552664
RUN  1000 , total integrated cost =  13239.628311552664
Improved over  1000  iterations in  148.01560898311436  seconds by  33.97933753633431  percent.
Problem in initial value trasfer:  Vmean_exc -56.695185163043085 -56.695185076028764
-------  75 0.5750000000000002 0.6750000000000004
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  350.6433804521056
Gradient descend method:  None
RUN  1 , total integrated cost =  45.71522390322435
RUN  2 , total integrated cost =  45.33054356121818
RUN  3 , total integrated cost =  45.30029909944468
RUN  4 , total integrated cost =  45.283997150575324
RUN  5 , total integrated cost =  45.26986254786969
RUN  6 , total integrated cost =  45.25806766915981
RUN  7 , total integrated cost =  45.24935632233549
RUN  8 , total integrated cost =  45.240001852324234
RUN  9 , total integrated cost =  45.233204129375785
RUN  10 , total integrated cost = 

In [ ]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

In [ ]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

In [ ]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

In [ ]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [ ]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

In [ ]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

In [ ]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

In [ ]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

In [ ]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)